## Tad

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.morphology import opening, closing, remove_small_objects
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi

def segment_wood(image_path, target_size=(512, 512)):
    image = imread(image_path)
    
    if image.ndim == 3:
        image_gray = rgb2gray(image)
    else:
        image_gray = image.astype(float) / 255.0
    
    # Resize
    image_small = resize(image, 
                         (target_size[0], target_size[1], image.shape[2]) 
                         if image.ndim == 3 else target_size, 
                         anti_aliasing=True)
    image_gray = resize(image_gray, target_size, anti_aliasing=True)
    
    # --- Step 1: Vertical grain suppression ---
    vertical_kernel = np.ones((21, 1))
    grain_estimate = opening(image_gray, vertical_kernel)
    grain_estimate = closing(grain_estimate, vertical_kernel)
    residual = np.abs(image_gray - grain_estimate)
    
    # --- Step 2: Otsu on residual ---
    thresh = threshold_otsu(residual)
    binary = residual > thresh
    
    # --- Step 3: Clean up with morphology ---
    # Fill small holes
    binary = ndi.binary_fill_holes(binary)
    # Remove small spurious objects
    binary = remove_small_objects(binary, min_size=200)
    
    return image_small, image_gray, residual, binary

def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    
    valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
    image_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(valid_extensions)
    ]
    
    print(f"Found {len(image_files)} images")
    
    for i, filename in enumerate(image_files):
        print(f"Processing {filename} ({i+1}/{len(image_files)})...", end=' ')
        
        image_path = os.path.join(input_folder, filename)
        
        try:
            image, image_gray, residual, binary = segment_wood(image_path)
            
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            
            axes[0].imshow(image_gray, cmap='gray')
            axes[0].set_title('Grayscale')
            axes[0].axis('off')
            
            axes[1].imshow(residual, cmap='gray')
            axes[1].set_title('After Grain Suppression')
            axes[1].axis('off')
            
            axes[2].imshow(binary, cmap='gray')
            axes[2].set_title('Otsu Threshold')
            axes[2].axis('off')
            
            axes[3].imshow(image_gray, cmap='gray')
            axes[3].contour(binary, [0.5], colors='r', linewidths=1.5)
            axes[3].set_title('Defects Overlaid')
            axes[3].axis('off')
            
            plt.suptitle(filename, fontsize=12)
            plt.tight_layout()
            
            output_name = os.path.splitext(filename)[0] + '_segmented.png'
            output_path = os.path.join(output_folder, output_name)
            plt.savefig(output_path, dpi=150, bbox_inches='tight')
            plt.close()
            
            print(f"done -> {output_name}")
            
        except Exception as e:
            print(f"ERROR: {e}")
            plt.close()

input_folder  = 'wood'
output_folder = 'Tad_Output'
process_folder(input_folder, output_folder)

## Nelson

In [2]:
import glob
from skimage.io import imread
from skimage.color import rgb2gray, rgba2rgb
from skimage.morphology import *
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
from skimage.filters import threshold_otsu, threshold_local, threshold_sauvola, sobel
import cv2
from skimage.feature import canny
from skimage.segmentation import felzenszwalb, slic, quickshift, watershed

In [17]:
SE = footprint_rectangle((10, 50))

path = "wood/*.png"
images = [cv2.imread(file) for file in glob.glob(path)]
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

example_img = cv2.imread("wood/001.png", cv2.IMREAD_GRAYSCALE)

equalized = clahe.apply(example_img)

threshold_result = equalized > threshold_otsu(equalized)

grain_removal = closing(threshold_result, SE)
grain_removal = opening(grain_removal, SE)

plt.figure(figsize=(15,15))
plt.subplot(2,3,1)
plt.axis("off")
plt.imshow(example_img, cmap="gray")
plt.savefig(f"Nelson's_Output/example_output_1.png", dpi=150, bbox_inches="tight", pad_inches=0)
plt.close()

plt.subplot(2,3,2)
plt.axis("off")
plt.imshow(equalized, cmap="gray")
plt.savefig(f"Nelson's_Output/example_output_2.png", dpi=150, bbox_inches="tight", pad_inches=0)
plt.close()

plt.subplot(2,3,3)
plt.axis("off")
plt.imshow(threshold_result, cmap="gray")
plt.savefig(f"Nelson's_Output/example_output_3.png", dpi=150, bbox_inches="tight", pad_inches=0)
plt.close()

plt.subplot(2,3,4)
plt.axis("off")
plt.imshow(grain_removal, cmap="gray")
plt.savefig(f"Nelson's_Output/example_output_4.png", dpi=150, bbox_inches="tight", pad_inches=0)
plt.close()

plt.subplot(2,3,5)
plt.axis("off")
plt.imshow(example_img, cmap="gray")
plt.contour(grain_removal, [0.5], colors='r', linewidths=1.5)
plt.savefig(f"Nelson's_Output/example_output_5.png", dpi=150, bbox_inches="tight", pad_inches=0)
plt.close()


for i, img in enumerate(images):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    equalized = clahe.apply(gray_image)

    threshold_result = equalized > threshold_otsu(equalized)
    
    grain_removal = closing(threshold_result, SE)
    grain_removal = opening(grain_removal, SE)


    plt.figure()
    plt.imshow(img_rgb)
    plt.contour(grain_removal, [0.5], colors='r', linewidths=1.5)
    plt.axis("off")
    plt.savefig(f"Nelson's_Output/output_{i:03d}.png", dpi=150, bbox_inches="tight", pad_inches=0)
    plt.close()

## Konner

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from skimage import color, filters, feature, morphology, measure

# Selected crack/gouge images from wood folder

image_files = [
    "007.png",
    "011.png",
    "013.png",
    "021.png",
    "023.png"
]


input_folder = "wood"


for filename in image_files:

    image_path = os.path.join(input_folder, filename)

    # Open selected image
    image = Image.open(image_path).convert("RGB")

    image_np = np.array(image)

    # Convert to grayscale
    gray = color.rgb2gray(image_np)

    # Blur image to reduce wood-grain noise using guassian
    blurred = filters.gaussian(
        gray,
        sigma=1
    )

    # Detect crack/gouge edges
    edges = feature.canny(
        blurred,
        sigma=2
    )

    # Connect crack/gouge lines
    closed = morphology.binary_closing(
        edges,
        morphology.disk(2)
    )

    # Remove small noise
    cleaned = morphology.remove_small_objects(
        closed,
        min_size=50
    )

    # Create output overlay
    output = image_np.copy()

    # Color detected regions red
    output[cleaned] = [255, 0, 0]

    # Display results
    plt.figure(figsize=(15,5))

    # Original image
    plt.subplot(1,3,1)
    plt.imshow(image_np)
    plt.title("Original")
    plt.axis("off")

    # Binary mask
    plt.subplot(1,3,2)
    plt.imshow(cleaned, cmap="gray")
    plt.title("Crack/Gouge Mask")
    plt.axis("off")

    # Overlay result
    plt.subplot(1,3,3)
    plt.imshow(output)
    plt.title("Detected Cracks/Gouges")
    plt.axis("off")

    plt.suptitle(filename)

    plt.show()